# GSM8K: Zero-shot → CoT → Few-shot → Retrieval → Self-consistency

This notebook walks through five progressively more involved prompting / inference strategies on the **GSM8K** grade-school math benchmark and measures how each one changes accuracy.

**The experiments, in order:**

1. **Experiment A — Zero-shot.** Just hand the model the question.
2. **Experiment B — Chain-of-thought (CoT).** Ask it to reason step by step, with the final answer in `\boxed{}`.
3. **Experiment C — Few-shot CoT.** Prepend `k ∈ {3, 5, 8}` solved exemplars from the train set, drawn randomly.
4. **Experiment D — Retrieval-based few-shot.** Replace the random exemplars with the `k` train problems most *similar* to each test question (by sentence-embedding cosine). We'll see this helps at small `k` and breaks down as `k` grows — and diagnose why.
5. **Experiment E — Self-consistency.** Sample `N ∈ {3, 5, 8}` reasoning chains at non-zero temperature and majority-vote the answer.

Along the way:

- **Answer extraction with `math-verify`** — we use the [`math-verify`](https://pypi.org/project/math-verify/) library to parse and compare mathematical expressions robustly (it handles `$3,500` vs `3500`, `\boxed{72}` vs `72`, fractions, etc.). The same library is reused for self-consistency voting so different surface forms of the same number cluster together.
- **A surprise at Experiment D** — retrieval is the textbook "next obvious upgrade" over random few-shot, and at `k=3` it works beautifully (+16 pp over random-3). But at `k=8` the lift evaporates. The diagnostic discussion in that section explains why — it's a useful lesson about when topical similarity *does* and *doesn't* predict transferable reasoning.

**Defaults:**
- `N_EVAL = 100` for fast iteration. Set to `None` (or larger) to evaluate on the full 1,319-question test set — error bars tighten roughly as `√n`.
- Model: `Qwen/Qwen2.5-0.5B-Instruct` — small enough to run anywhere, weak enough that prompting effects are clearly visible. Swap `MODEL_ID` for a bigger model if you want a stronger ceiling.
- Assumes a CUDA GPU is available; `device_map="auto"` will fall back to CPU but it'll be slow.

## 1. Install dependencies

We need:
- `transformers` + `accelerate` — model loading and generation.
- `datasets` — to pull GSM8K from the Hub.
- `math-verify` — robust math-answer comparison.
- `torch` — already present in most GPU images.

In [1]:
!pip install -q transformers accelerate datasets math-verify

## 2. Imports & determinism

Greedy decoding (`do_sample=False`) plus fixed seeds make the experiments reproducible. The seed mostly matters for the few-shot section where we sample example problems from the train set.

In [2]:
import re
import random
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from math_verify import parse, verify
from tqdm.auto import tqdm

random.seed(0)
torch.manual_seed(0)

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

torch: 2.8.0+cu128 | cuda available: True


## 3. Load GSM8K

GSM8K has two splits:
- `train` — 7,473 problems, each with a worked solution. We'll pull few-shot examples from here.
- `test` — 1,319 problems. This is what we score on.

Each item is `{'question': str, 'answer': str}` where the `answer` field contains the full chain of reasoning followed by `#### <final_number>` on the last line. We'll extract the final number for grading.

**To run on the full test set:** change `N_EVAL = 100` to `N_EVAL = None`.

In [3]:
N_EVAL = 100   # set to None to evaluate on the full 1,319-item test set

ds = load_dataset("openai/gsm8k", "main")
train_set = ds["train"]
test_set = ds["test"] if N_EVAL is None else ds["test"].select(range(N_EVAL))

print(f"Train: {len(train_set)} | Test (evaluating on): {len(test_set)}")
print("\nExample question:\n", test_set[0]["question"])
print("\nExample answer (note the '#### N' at the end):\n", test_set[0]["answer"])

Train: 7473 | Test (evaluating on): 100

Example question:
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Example answer (note the '#### N' at the end):
 Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


## 4. Load the model

We use `Qwen/Qwen2.5-0.5B-Instruct` — the general (non-code-specialised) instruction-tuned 0.5B model:

- 0.5B params → fits in <4 GB of VRAM in bf16.
- 32k context window → plenty of room for many-shot prompts.
- Instruction-tuned, follows `system` / `user` / `assistant` chat roles via the tokenizer's chat template.

We also configure left-padding because we'll do **batched** generation. Decoder-only LMs need padding on the **left** so the right edge (where new tokens get appended) lines up across the batch.

In [4]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",      # bf16 on Ampere+, fp16 otherwise
    device_map="auto",       # uses GPU if available, else CPU
)
model.eval()

CTX_LEN = getattr(model.config, "max_position_embeddings", 32768)
print(f"Loaded {MODEL_ID}")
print(f"  device: {model.device} | dtype: {next(model.parameters()).dtype}")
print(f"  context length: {CTX_LEN}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct
  device: cuda:0 | dtype: torch.bfloat16
  context length: 32768


## 5. Grading with `math-verify`

Hand-coding answer extraction is fragile — models will output `\boxed{72}`, `**72**`, `"The answer is 72."`, `$72.00$`, `72 dollars`, etc. The `math-verify` library handles this:

- `parse(text)` extracts a mathematical expression from arbitrary text (LaTeX, plain numbers, fractions, sets, …).
- `verify(gold, pred)` returns `True` iff they're mathematically equivalent.

We wrap the gold GSM8K answer in `\boxed{...}` so the parser treats it as a single canonical expression, then compare against whatever the model produced.

In [5]:
def extract_gold(answer_text: str) -> str:
    """Pull the final numeric answer from a GSM8K 'answer' field (after the '#### ' marker)."""
    m = re.search(r"####\s*(.+)", answer_text)
    return m.group(1).strip() if m else answer_text.strip()

def is_correct(pred_text: str, gold_text: str) -> bool:
    """True iff the model output contains the same mathematical answer as `gold_text`."""
    try:
        gold = parse(f"\\boxed{{{gold_text}}}")
        pred = parse(pred_text)
        return bool(verify(gold, pred))
    except Exception:
        return False

# Sanity check the grader
assert is_correct("The answer is \\boxed{72}.", "72")
assert is_correct("So the final answer is 72.", "72")
assert not is_correct("\\boxed{18}", "72")
assert is_correct("$3,500", "3500")
print("grader sanity checks passed")

grader sanity checks passed


## 6. Batched generation helper

A small wrapper around `model.generate` that:
- Batches prompts for throughput.
- Decodes **only the newly-generated tokens** (slices past the input length), so we don't have to strip the prompt back out of the output.
- Uses greedy decoding for reproducibility.

We also include a token-budget check: GSM8K solutions are typically <300 generated tokens, but few-shot prompts can grow large, so we make sure `prompt_tokens + max_new_tokens` stays under the model's context window. If it doesn't, you'd see silent truncation of the prompt and wrong answers — exactly the bug we want to avoid.

In [6]:
def build_prompt(messages):
    """Apply the model's chat template and append the assistant-turn cue."""
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def check_token_budget(prompts, max_new_tokens, label=""):
    lengths = [len(tokenizer.encode(p)) for p in prompts]
    max_p, mean_p = max(lengths), sum(lengths) / len(lengths)
    total = max_p + max_new_tokens
    print(f"  [{label}] prompt tokens — max: {max_p}, mean: {mean_p:.0f} | +{max_new_tokens} gen → worst-case {total} (ctx {CTX_LEN})")
    if total >= CTX_LEN:
        raise RuntimeError(f"Prompt+gen ({total}) exceeds context window ({CTX_LEN}); reduce shots or max_new_tokens.")

@torch.no_grad()
def generate(prompts, max_new_tokens=512, batch_size=8):
    outputs = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="generate"):
        batch = prompts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        new_tokens = out[:, inputs["input_ids"].shape[1] :]
        outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    return outputs

def evaluate(prompts, eval_set, max_new_tokens, label):
    check_token_budget(prompts, max_new_tokens, label=label)
    preds = generate(prompts, max_new_tokens=max_new_tokens)
    correct = sum(
        is_correct(pred, extract_gold(ex["answer"])) for pred, ex in zip(preds, eval_set)
    )
    acc = correct / len(eval_set)
    print(f"  [{label}] accuracy: {correct}/{len(eval_set)} = {acc:.1%}")
    return acc, preds

## 7. Experiment A — Zero-shot, no structure

The simplest possible prompt: just hand the model the question with a minimal system message. This is our floor — everything else has to beat it.

We allow 256 new tokens; even without an explicit CoT instruction, instruction-tuned models will usually produce *some* reasoning, but the final answer can be anywhere in the output and in any format. That's exactly why we need `math-verify` to grade.

In [7]:
SYS_ZERO = "You are a helpful assistant. Answer the user's math question."

zero_shot_prompts = [
    build_prompt([
        {"role": "system", "content": SYS_ZERO},
        {"role": "user", "content": ex["question"]},
    ])
    for ex in test_set
]

print("=== Example zero-shot prompt ===")
print(zero_shot_prompts[0])
print("=" * 40)

acc_zero, preds_zero = evaluate(zero_shot_prompts, test_set, max_new_tokens=256, label="zero-shot")

=== Example zero-shot prompt ===
<|im_start|>system
You are a helpful assistant. Answer the user's math question.<|im_end|>
<|im_start|>user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?<|im_end|>
<|im_start|>assistant

  [zero-shot] prompt tokens — max: 154, mean: 85 | +256 gen → worst-case 410 (ctx 32768)


generate:   0%|          | 0/13 [00:00<?, ?it/s]

  [zero-shot] accuracy: 27/100 = 27.0%


## 8. Experiment B — Structured CoT prompt

Two small changes, big effect:
1. **Explicit chain-of-thought instruction** — tell the model to reason step by step before answering. This is the canonical ["Let's think step by step"](https://arxiv.org/abs/2205.11916) trick.
2. **Structured final answer** — require the answer in `\boxed{...}` on the last line. This gives `math-verify` an unambiguous target and avoids cases where the model trails off with extra prose after the answer.

We bump `max_new_tokens` to 512 because the reasoning trace is longer. GSM8K solutions typically fit comfortably in 300–400 tokens, so 512 leaves headroom while staying small enough that batching is still fast.

In [8]:
SYS_COT = (
    "You are a careful math tutor. Solve the problem step by step. "
    "Show your reasoning clearly, then write the FINAL numeric answer "
    "inside \\boxed{} on the last line. Use only digits inside the box "
    "(no units, no commas)."
)

cot_prompts = [
    build_prompt([
        {"role": "system", "content": SYS_COT},
        {"role": "user", "content": ex["question"]},
    ])
    for ex in test_set
]

print("=== Example CoT prompt ===")
print(cot_prompts[0])
print("=" * 40)

acc_cot, preds_cot = evaluate(cot_prompts, test_set, max_new_tokens=512, label="CoT (0-shot)")

=== Example CoT prompt ===
<|im_start|>system
You are a careful math tutor. Solve the problem step by step. Show your reasoning clearly, then write the FINAL numeric answer inside \boxed{} on the last line. Use only digits inside the box (no units, no commas).<|im_end|>
<|im_start|>user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?<|im_end|>
<|im_start|>assistant

  [CoT (0-shot)] prompt tokens — max: 188, mean: 119 | +512 gen → worst-case 700 (ctx 32768)


generate:   0%|          | 0/13 [00:00<?, ?it/s]

  [CoT (0-shot)] accuracy: 42/100 = 42.0%


### Inspect a couple of CoT outputs

A quick eyeball check that the format is what we asked for. If the model is ignoring `\boxed{}` you'll see it here, and you'd tighten the system prompt.

In [9]:
for idx in [0, 1, 2]:
    gold = extract_gold(test_set[idx]["answer"])
    print(f"--- Q{idx} (gold={gold}) ---")
    print("Q:", test_set[idx]["question"][:120], "...")
    print("A:", preds_cot[idx])
    print("correct:", is_correct(preds_cot[idx], gold))
    print()

--- Q0 (gold=18) ---
Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every da ...
A: To determine how much Janet makes at the farmers' market each day, we need to follow these steps:

1. Calculate the total number of eggs laid by the ducks in one day.
2. Determine how many eggs are eaten in one day.
3. Subtract the number of eggs eaten from the total number of eggs to find out how many eggs are sold.
4. Calculate the revenue from selling the eggs.

First, let's calculate the total number of eggs laid by the ducks in one day:
\[ 16 \text{ eggs/day} + 3 \text{ eggs (breakfast)} + 4 \text{ eggs (baking)} = 23 \text{ eggs/day} \]

Next, we know that Janet eats 3 eggs for breakfast and 4 eggs baking, so the total number of eggs eaten in one day is:
\[ 3 \text{ eggs (breakfast)} + 4 \text{ eggs (baking)} = 7 \text{ eggs/day} \]

Now, we subtract the number of eggs eaten from the total number of eggs to find out how many eggs 

## 9. Experiment C — Few-shot CoT (k = 3, 5, 8)

Now we prepend `k` solved examples from the **train** split. Each example is a `(question, worked-solution + \boxed{answer})` turn pair, formatted as the conversation history the model would have produced if it followed our CoT format perfectly.

Design choices:
- **Fixed, randomly sampled exemplars** (deterministic via `random.seed(42)`). Same `k=8` pool is reused for the smaller `k` runs (taking the first 3, 5, 8) so each step purely adds context.
- We convert the train answer's `"...reasoning... #### 18"` into `"...reasoning... \\boxed{18}"` so the model sees consistent formatting with what we want back.
- We're isolating one axis here — *how many shots*. In the **next experiment** (Experiment D) we'll change *which* shots, picking nearest-neighbours instead of a random pool, and see what that does.

**Token budget:** Each GSM8K train solution is ~120–250 tokens. 8 shots ≈ 2–3k tokens of context, plus the test question and 512 generation tokens — well within Qwen's 32k window. The `check_token_budget` call inside `evaluate` will raise if anything overflows.

In [10]:
random.seed(42)
MAX_SHOTS = 8
shot_indices = random.sample(range(len(train_set)), MAX_SHOTS)
shot_pool = [train_set[i] for i in shot_indices]

def format_assistant(answer_field: str) -> str:
    """Convert a GSM8K answer ('reasoning ... #### 18') into our CoT+\\boxed{} format."""
    reasoning, _, final = answer_field.rpartition("####")
    # GSM8K answers contain '<<...>>' calculator annotations — strip them for cleaner exemplars.
    reasoning = re.sub(r"<<.*?>>", "", reasoning).strip()
    return f"{reasoning}\n\\boxed{{{final.strip()}}}"

def build_fewshot_prompt(question: str, examples):
    msgs = [{"role": "system", "content": SYS_COT}]
    for ex in examples:
        msgs.append({"role": "user", "content": ex["question"]})
        msgs.append({"role": "assistant", "content": format_assistant(ex["answer"])})
    msgs.append({"role": "user", "content": question})
    return build_prompt(msgs)

# Sanity check: print a 3-shot prompt for the first test question
demo = build_fewshot_prompt(test_set[0]["question"], shot_pool[:3])
print("=== 3-shot prompt (truncated) ===")
print(demo[:1500])
print("...")
print(demo[-400:])

=== 3-shot prompt (truncated) ===
<|im_start|>system
You are a careful math tutor. Solve the problem step by step. Show your reasoning clearly, then write the FINAL numeric answer inside \boxed{} on the last line. Use only digits inside the box (no units, no commas).<|im_end|>
<|im_start|>user
For every 12 cans you recycle, you receive $0.50, and for every 5 kilograms of newspapers, you receive $1.50. If your family collected 144 cans and 20 kilograms of newspapers, how much money would you receive?<|im_end|>
<|im_start|>assistant
There are 144/12 = 12 sets of 12 cans that the family collected.
So, the family would receive $0.50 x 12 = $6 for the cans.
There are 20/5 = 4 sets of 5 kilograms of newspapers that the family collected.
So, the family would receive $1.50 x 4 = $6 for the newspapers.
Therefore, the family would receive a total of $6 + $6 = $12.
\boxed{12}<|im_end|>
<|im_start|>user
Betty picked 16 strawberries. Matthew picked 20 more strawberries than Betty and twice as many 

In [11]:
fewshot_results = {}
for k in [3, 5, 8]:
    examples = shot_pool[:k]
    prompts = [build_fewshot_prompt(ex["question"], examples) for ex in test_set]
    acc, _ = evaluate(prompts, test_set, max_new_tokens=512, label=f"{k}-shot CoT")
    fewshot_results[k] = acc

  [3-shot CoT] prompt tokens — max: 639, mean: 570 | +512 gen → worst-case 1151 (ctx 32768)


generate:   0%|          | 0/13 [00:00<?, ?it/s]

  [3-shot CoT] accuracy: 30/100 = 30.0%
  [5-shot CoT] prompt tokens — max: 893, mean: 824 | +512 gen → worst-case 1405 (ctx 32768)


generate:   0%|          | 0/13 [00:00<?, ?it/s]

  [5-shot CoT] accuracy: 40/100 = 40.0%
  [8-shot CoT] prompt tokens — max: 1473, mean: 1404 | +512 gen → worst-case 1985 (ctx 32768)


generate:   0%|          | 0/13 [00:00<?, ?it/s]

  [8-shot CoT] accuracy: 34/100 = 34.0%


## Experiment D — Retrieval-based few-shot CoT

So far our few-shot examples were a fixed, **random** pool of 8 train problems — the same exemplars for every test question. A natural next step: for each test question, find the *most similar* train problems and use those as exemplars. The intuition is that examples close to the target in problem structure (rates, ratios, ages, two-step word problems, …) give the model a stronger template to follow.

How we do it:

1. Embed all 7,473 train questions with a small sentence-transformer (`all-MiniLM-L6-v2`, 384-d).
2. Embed each of the test questions the same way.
3. For each test question, compute cosine similarity to every train question and pick the top-`k`.
4. **Order the exemplars least-similar → most-similar**, so the most relevant example is right next to the actual question (decoder-only models pay more attention to nearby context).
5. Run at the same `k ∈ {3, 5, 8}` we used for random few-shot, so the comparison is apples-to-apples — only the *selection rule* changes between the random pool and the retrieved pool.
6. Use a larger `max_new_tokens` budget than the random-pool run. With very similar exemplars, the model tends to imitate the (often verbose) CoT style of its matched neighbors, and 512 generation tokens can get cut off before the `\boxed{}` final answer. We bump to 1024 and print a truncation diagnostic.

The textbook claim is that retrieval should help across the board. In practice we'll see something more interesting — it helps at small `k` and stops helping at larger `k`. The diagnostic discussion after the run unpacks why.

In [12]:
!pip install -q sentence-transformers

In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Use the same GPU as the main model (falls back to CPU if no CUDA)
embed_device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=embed_device)

train_questions = train_set["question"]
test_questions = [ex["question"] for ex in test_set]

# Encode once. ~7k train items takes ~30s on a GPU, a couple of minutes on CPU.
train_emb = embedder.encode(
    train_questions, batch_size=128, convert_to_numpy=True,
    show_progress_bar=True, normalize_embeddings=True,
)
test_emb = embedder.encode(
    test_questions, batch_size=128, convert_to_numpy=True,
    show_progress_bar=True, normalize_embeddings=True,
)

print("train embeddings:", train_emb.shape)
print("test  embeddings:", test_emb.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

train embeddings: (7473, 384)
test  embeddings: (100, 384)


In [14]:
RETRIEVAL_MAX_NEW = 1024  # bumped from 512 — matched exemplars often cause longer chains

sim = test_emb @ train_emb.T                          # (n_test, n_train)
top_k_indices = np.argsort(sim, axis=1)[:, -8:]       # ascending → nearest LAST (we take suffixes for smaller k)

# Peek at the retrieved neighbors for the first test question — same surface,
# different math, every time. This is exactly the "topical not structural"
# failure mode the discussion below covers.
print("Test Q0:", test_questions[0][:120], "...\n")
print("Top-8 train neighbors (least → most similar):")
for j in top_k_indices[0]:
    print(f"  sim={sim[0, int(j)]:.3f}  |  {train_questions[int(j)][:100]} ...")
print()

# Run retrieval at k = 3, 5, 8 for parity with the random few-shot section above.
retrieval_results = {}
retrieval_preds = {}
for k in [3, 5, 8]:
    sub_indices = top_k_indices[:, -k:]
    prompts = []
    for i, ex in enumerate(test_set):
        neighbors = [train_set[int(j)] for j in sub_indices[i]]
        prompts.append(build_fewshot_prompt(ex["question"], neighbors))
    acc, preds = evaluate(
        prompts, test_set, max_new_tokens=RETRIEVAL_MAX_NEW,
        label=f"{k}-shot CoT (retrieval)"
    )
    retrieval_results[k] = acc
    retrieval_preds[k] = preds

# Keep the k=8 predictions and accuracy under their old names so the summary
# cell and the diagnostics below can reference them directly.
preds_retr8 = retrieval_preds[8]
acc_retr8   = retrieval_results[8]

# --- Truncation diagnostic (on the k=8 run) ----------------------------------
# These three numbers rule truncation in or out as the cause when retrieval
# underperforms: how many outputs are missing the final \boxed{...}, how many
# hit the generation cap, and the output length distribution.
no_box      = sum(1 for p in preds_retr8 if "\\boxed{" not in p)
out_lengths = [len(tokenizer.encode(p)) for p in preds_retr8]
hit_cap     = sum(1 for L in out_lengths if L >= RETRIEVAL_MAX_NEW - 5)
print(f"\nDiagnostics for the k=8 retrieval run:")
print(f"  outputs WITHOUT \\boxed{{...}}   : {no_box}/{len(preds_retr8)}  (truncated or format failure)")
print(f"  outputs at the token cap       : {hit_cap}/{len(preds_retr8)}")
print(f"  output length (tokens)         : mean {sum(out_lengths)/len(out_lengths):.0f}, max {max(out_lengths)}")

# Show the tail of the first failure so you can eyeball whether it's mid-CoT truncation
for i, p in enumerate(preds_retr8):
    gold = extract_gold(test_set[i]["answer"])
    if not is_correct(p, gold):
        print(f"\n--- First failure: Q{i} (gold={gold}) — last 350 chars of output ---")
        print(p[-350:])
        break


Test Q0: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every da ...

Top-8 train neighbors (least → most similar):
  sim=0.706  |  Samantha bought a crate of 30 eggs for $5. If she decides to sell each egg for 20 cents, how many eg ...
  sim=0.712  |  Alan went to the market and bought 20 eggs at the price of $2 per egg. He bought 6 chickens for the  ...
  sim=0.718  |  Mark has an egg farm. His farm supplies one store with 5 dozen eggs and another store with 30 eggs e ...
  sim=0.718  |  Krista started raising chickens. She has 10 hens who lay eggs. She sells the eggs for $3 a dozen. In ...
  sim=0.722  |  A farmer has 46 chickens. Each chicken gives him 6 eggs a week. If he sells a dozen eggs for $3, how ...
  sim=0.731  |  Peter has 15 birds. 1/3 are ducks. The rest are chickens and require special feed that costs $2 per  ...
  sim=0.761  |  Jane runs a small farm.  She has 10 chickens that lay 6 eggs each per week. S

generate:   0%|          | 0/13 [00:00<?, ?it/s]

  [3-shot CoT (retrieval)] accuracy: 46/100 = 46.0%
  [5-shot CoT (retrieval)] prompt tokens — max: 1488, mean: 919 | +1024 gen → worst-case 2512 (ctx 32768)


generate:   0%|          | 0/13 [00:00<?, ?it/s]

  [5-shot CoT (retrieval)] accuracy: 43/100 = 43.0%
  [8-shot CoT (retrieval)] prompt tokens — max: 2083, mean: 1424 | +1024 gen → worst-case 3107 (ctx 32768)


generate:   0%|          | 0/13 [00:00<?, ?it/s]

  [8-shot CoT (retrieval)] accuracy: 36/100 = 36.0%

Diagnostics for the k=8 retrieval run:
  outputs WITHOUT \boxed{...}   : 17/100  (truncated or format failure)
  outputs at the token cap       : 0/100
  output length (tokens)         : mean 110, max 384

--- First failure: Q0 (gold=18) — last 350 chars of output ---
She eats 16 - 3 - 4 = 19 eggs a day.
So she sells 19 * 2 = 38 eggs a day.

Therefore, Janet makes 38 * $2 = $76 every day at the farmers' market.

\boxed{76}


### Retrieval helps at small `k`, but breaks down beyond ~3 examples

| k  | random pool | retrieval (top-k) | Δ (retrieval − random) |
|---:|---:|---:|---:|
| 3  | 30.0%       | **46.0%**         | **+16.0 pp**           |
| 5  | 40.0%       | 43.0%             | +3.0                   |
| 8  | 34.0%       | 36.0%             | +2.0                   |

At `k=3` retrieval is the cleanest single-pass win in the notebook — it beats random-3 by **+16 pp** and zero-shot CoT (42%) by **+4 pp**. That's the textbook "matched exemplars give the model a better template" claim, and it holds. But the lift fades as `k` grows: by `k=8`, retrieval and random are tied.

The intuition is straightforward. Look at the Top-8 retrieved neighbours printed above for the Janet question — they're *all* about eggs, chickens, or farms. The embedder did its job: it pulled the most topically-similar train problems. But that means **the retrieved set isn't diverse** — it's eight near-clones of each other, none of which actually require Janet's specific arithmetic (`16 - 3 - 4 = 9` eggs, then `9 × $2 = $18`).

At small `k`, that lack of diversity is fine: one or two on-topic exemplars give the model a useful template-shaped hint without flooding the context. At larger `k`, the same lack of diversity becomes a problem — the model sees a wall of egg-related problems all solving slightly different math, and gets confused about which operation pattern to copy. With random exemplars at `k=8` the model can tell none of them apply and falls back on its own CoT; with eight near-clones it tries to imitate them and ends up averaging incompatible templates.

So the practical lesson is **tune `k` carefully when retrieving**. The win at `k=3` is real and worth taking — accuracy goes up and the prompt is shorter. But more isn't better here, because what the model needs is *variety* in its exemplars, and similarity-based retrieval gives the opposite of that.

On tasks where surface similarity *does* predict the right solution method — customer support / FAQ, code-pattern completion, format-imitation — retrieval typically scales well to larger `k`, because each additional matched example adds real, transferable signal. GSM8K is the canonical *intermediate* case: small-`k` retrieval helps, then the signal saturates.

In the next experiment we'll switch levers entirely: instead of changing *what* we condition the model on, we **sample multiple reasoning chains** from the same prompt and majority-vote. As we'll see, that lift is even bigger than retrieval at this scale.

## Experiment E — Self-consistency (majority vote)

Greedy decoding picks the single most-likely reasoning chain. But there are many valid chains that arrive at the right answer, and many subtly-wrong chains that arrive at the same wrong answer. **Self-consistency** ([Wang et al., 2022](https://arxiv.org/abs/2203.11171)) is the simple-and-effective idea: sample several reasoning chains at non-zero temperature, extract the final answer from each, and **majority-vote** the answer.

Implementation notes:

- Sampling with `do_sample=True, temperature=0.7, top_p=0.95, num_return_sequences=N`.
- **For voting we re-use `math-verify`** — the same library we already use for grading. The previous draft of this cell hand-rolled a regex extractor and a string normalizer, but that's exactly the brittleness `math-verify` was built to avoid: it would put `\frac{1}{2}` and `0.5` (or `$3,500` and `3500`) into different vote buckets even though they're the same answer. Instead, we `parse()` each sampled chain into a math expression and **cluster** the candidates by pairwise `verify()` equivalence; the majority vote is a representative of the largest cluster.
- We apply self-consistency on top of the **CoT zero-shot** prompt to isolate the effect of sampling+voting. The same technique stacks cleanly with the few-shot or retrieval-based prompts above (try it as a follow-up).

Cost scales linearly in N: with N=8 samples per question this run does 8× the generation work of the greedy CoT pass.

In [15]:
@torch.no_grad()
def generate_samples(prompts, n_samples=8, max_new_tokens=512,
                     temperature=0.7, top_p=0.95, batch_size=4):
    """For each prompt return a list of n_samples decoded completions."""
    results = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="sample"):
        batch = prompts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            num_return_sequences=n_samples,
            pad_token_id=tokenizer.pad_token_id,
        )
        new_tokens = out[:, inputs["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        # HF groups outputs by prompt: prompt 0 → first n_samples entries, etc.
        for p_idx in range(len(batch)):
            results.append(decoded[p_idx * n_samples : (p_idx + 1) * n_samples])
    return results

def vote_clusters(samples):
    """Cluster sampled chains by math-verify equivalence.

    Returns a list of (representative_text, vote_count), largest cluster first.
    Samples whose `parse()` returns an empty list (no math found / truncated mid-thought)
    are skipped — they can't vote.
    """
    parsed = []
    for s in samples:
        try:
            p = parse(s)
            if p:                      # non-empty list = something parseable
                parsed.append((s, p))
        except Exception:
            continue

    clusters = []  # each entry: [representative_text, representative_parsed, count]
    for s, p in parsed:
        placed = False
        for c in clusters:
            try:
                if verify(c[1], p):
                    c[2] += 1
                    placed = True
                    break
            except Exception:
                pass
        if not placed:
            clusters.append([s, p, 1])

    clusters.sort(key=lambda c: -c[2])
    return [(c[0], c[2]) for c in clusters]

def majority_vote(samples):
    """Return a representative completion from the most-voted math-verify cluster (or '' if none parseable)."""
    cl = vote_clusters(samples)
    return cl[0][0] if cl else ""


In [16]:
# 95% Wald CI for a binomial proportion — used here and in the summary cell below.
def ci95(p, n):
    return 1.96 * (p * (1 - p) / n) ** 0.5 if 0 < p < 1 else 0.0

check_token_budget(cot_prompts, max_new_tokens=512, label="SC")

# We want to see how SC accuracy scales with N (the number of sampled chains).
# Strategy: sample N=8 once, then evaluate the majority vote over the first N
# of those samples for each N ∈ {3, 5, 8}. This re-uses the same 8 generations
# instead of paying 3+5+8 = 16x the cost.
N_MAX = 8
torch.manual_seed(0)
sc_samples = generate_samples(cot_prompts, n_samples=N_MAX, max_new_tokens=512)

sc_results = {}
for N in [3, 5, 8]:
    correct = 0
    for samples, ex in zip(sc_samples, test_set):
        voted = majority_vote(samples[:N])  # take the first N of the 8 sampled chains
        if is_correct(voted, extract_gold(ex["answer"])):
            correct += 1
    acc = correct / len(test_set)
    sc_results[N] = acc
    print(f"  [SC majority vote (N={N})] {correct}/{len(test_set)} = {acc:.1%} ± {ci95(acc, len(test_set)):.1%}")

# Keep N=8 under the legacy name for the summary cell.
acc_sc = sc_results[8]
N_SAMPLES = N_MAX
print(f"\n  (greedy CoT on {MODEL_ID} for comparison: {acc_cot:.1%} ± {ci95(acc_cot, len(test_set)):.1%})")

# Peek at a question where SC fixed a greedy-CoT failure, showing the cluster sizes.
for i in range(len(test_set)):
    gold = extract_gold(test_set[i]["answer"])
    greedy_ok = is_correct(preds_cot[i], gold)
    sc_ok     = is_correct(majority_vote(sc_samples[i]), gold)
    if sc_ok and not greedy_ok:
        cl = vote_clusters(sc_samples[i])
        sizes = [c[1] for c in cl]
        print(f"\nExample SC fixed Q{i} (gold={gold}): {len(cl)} math-verify cluster(s), sizes = {sizes}")
        break


  [SC] prompt tokens — max: 188, mean: 119 | +512 gen → worst-case 700 (ctx 32768)


sample:   0%|          | 0/25 [00:00<?, ?it/s]

  [SC majority vote (N=3)] 44/100 = 44.0% ± 9.7%


  [SC majority vote (N=5)] 49/100 = 49.0% ± 9.8%


  [SC majority vote (N=8)] 54/100 = 54.0% ± 9.8%

  (greedy CoT on Qwen/Qwen2.5-0.5B-Instruct for comparison: 42.0% ± 9.7%)

Example SC fixed Q0 (gold=18): 4 math-verify cluster(s), sizes = [4, 2, 1, 1]


## 10. Summary

All five conditions side by side on `Qwen2.5-0.5B-Instruct` at n=100. The pedagogical progression is *simplest prompting → more structured prompting → more in-context information → sampling-based methods*. Each technique trades off something different — compute, prompt length, exemplar diversity — and the table makes the picture concrete:

- **Experiments A–C** change what we put in the prompt: a bare question, a CoT instruction, then `k` solved exemplars.
- **Experiment D** changes *which* exemplars: nearest-neighbours instead of a random pool. Helps at small `k`, breaks down at larger `k` — for the reasons explained above.
- **Experiment E** changes *how* we decode: many samples + majority vote instead of one greedy chain. This is the biggest single lift in the notebook.

Reporting Wald 95% CIs alongside each number — at n=100 these are wide (~±10 pp around 50% accuracy), so treat 3–5 pp gaps as suggestive rather than decisive. Bump `N_EVAL` (cell at the top) to 500 or `None` to tighten the bars.

In [17]:
print(f"Reporting accuracy ± 95% Wald CI (n = {len(test_set)})\n")
print(f"--- {MODEL_ID} ---")
print(f"  zero-shot (no structure)            : {acc_zero:.1%} ± {ci95(acc_zero, len(test_set)):.1%}")
print(f"  CoT, 0-shot, \\boxed{{}}             : {acc_cot:.1%} ± {ci95(acc_cot, len(test_set)):.1%}")
for k, acc in fewshot_results.items():
    print(f"  {k}-shot CoT (random pool)          : {acc:.1%} ± {ci95(acc, len(test_set)):.1%}")
for k, acc in retrieval_results.items():
    print(f"  {k}-shot CoT (retrieval, top-k)     : {acc:.1%} ± {ci95(acc, len(test_set)):.1%}")
for N, acc in sc_results.items():
    print(f"  CoT + self-consistency (N={N})       : {acc:.1%} ± {ci95(acc, len(test_set)):.1%}")


Reporting accuracy ± 95% Wald CI (n = 100)

--- Qwen/Qwen2.5-0.5B-Instruct ---
  zero-shot (no structure)            : 27.0% ± 8.7%
  CoT, 0-shot, \boxed{}             : 42.0% ± 9.7%
  3-shot CoT (random pool)          : 30.0% ± 9.0%
  5-shot CoT (random pool)          : 40.0% ± 9.6%
  8-shot CoT (random pool)          : 34.0% ± 9.3%
  3-shot CoT (retrieval, top-k)     : 46.0% ± 9.8%
  5-shot CoT (retrieval, top-k)     : 43.0% ± 9.7%
  8-shot CoT (retrieval, top-k)     : 36.0% ± 9.4%
  CoT + self-consistency (N=3)       : 44.0% ± 9.7%
  CoT + self-consistency (N=5)       : 49.0% ± 9.8%
  CoT + self-consistency (N=8)       : 54.0% ± 9.8%
